<a href="https://colab.research.google.com/github/rudra629/ml-internship-flyrank/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Lane: Refresh / Content Opportunity Scoring

What one row means: A single web page's search performance aggregated over a 1-month period.

Which table(s): fact_content_daily_performance (for metrics) joined with dim_clients (for access verification).

Time window: A mid-panel month, specifically March 2026 (month=2026-03), to avoid leakage into the final test month (June).

Label/Proxy: needs_refresh (1 if impressions > median AND CTR < median, else 0).

Deliberate Exclusion: I am excluding pages with < 1000 impressions to eliminate statistical noise and low-signal edge cases.

In [4]:
import duckdb
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score
from google.colab import userdata

# 1. Authenticate using Colab Secrets (No hardcoded tokens!)
hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# --- THE THREE QUERIES ---

# Query 1: The Grain (One row = One page aggregated over March 2026)
print("--- Query 1: The Grain ---")
q1 = f"""
SELECT content_hash_id, COUNT(*) as days_recorded
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY content_hash_id
LIMIT 3
"""
print(con.sql(q1).df())

# Query 2: Row Count & Date Span for this slice
print("\n--- Query 2: Date Span & Row Count ---")

# 1. Ask DuckDB for all the column names to find the exact date column
cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')").df()
date_col = [c for c in cols['column_name'] if 'date' in c.lower() or c.lower() == 'day'][0]

# 2. Run the query using the dynamically found column name
q2 = f"""
SELECT
    MIN({date_col}) as start_date,
    MAX({date_col}) as end_date,
    COUNT(*) as total_raw_rows
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
"""
print(con.sql(q2).df())

# Query 3: Availability (IS TRUE)
print("\n--- Query 3: Availability (IS TRUE) ---")
q3 = f"""
SELECT COUNT(DISTINCT f.content_hash_id) as valid_pages
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') f
JOIN read_parquet('{REL}/dim_clients.parquet') c
  ON f.client_hash_id = c.client_hash_id
WHERE c.has_gsc_access IS TRUE
"""
print(con.sql(q3).df())

# --- THE FIVE FEATURES & THE TRAP ---

print("\n--- Feature Engineering & The Leakage Trap ---")
# Build the feature frame
query_features = f"""
SELECT
    f.content_hash_id,
    SUM(f.gsc_clicks) as total_clicks,
    SUM(f.gsc_impressions) as total_impressions,
    AVG(f.gsc_avg_position) as avg_position,
    (SUM(f.gsc_clicks)*1.0 / NULLIF(SUM(f.gsc_impressions), 0)) as true_ctr
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') f
JOIN read_parquet('{REL}/dim_clients.parquet') c ON f.client_hash_id = c.client_hash_id
WHERE c.has_gsc_access IS TRUE
GROUP BY f.content_hash_id
HAVING total_impressions > 1000
"""
df = con.sql(query_features).df().dropna()

# Create target label
ctr_threshold = df['true_ctr'].median()
df['needs_refresh'] = ((df['total_impressions'] > df['total_impressions'].median()) &
                       (df['true_ctr'] < ctr_threshold)).astype(int)

# THE TRAP: Including 'true_ctr' as a feature when it was literally used to calculate the label
features_with_leak = ['total_impressions', 'avg_position', 'total_clicks', 'true_ctr']
X_leak = df[features_with_leak]
y = df['needs_refresh']

model_leak = DecisionTreeClassifier(max_depth=3, random_state=42)
model_leak.fit(X_leak, y)
preds_leak = model_leak.predict(X_leak)
print(f"Leaked Score (Precision): {precision_score(y, preds_leak):.2f} <- Unrealistic! Model learned the math, not the pattern.")

# REMOVE THE TRAP
honest_features = ['total_impressions', 'avg_position', 'total_clicks']
X_honest = df[honest_features]
model_honest = DecisionTreeClassifier(max_depth=3, random_state=42)
model_honest.fit(X_honest, y)
preds_honest = model_honest.predict(X_honest)
print(f"Honest Score (Precision): {precision_score(y, preds_honest):.2f} <- Safe, no target leakage.")

--- Query 1: The Grain ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id  days_recorded
0  content_7a105f548d9c6916             31
1  content_a3ea9792f793ec72             31
2  content_36c36abc7650d7af             31

--- Query 2: Date Span & Row Count ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  start_date   end_date  total_raw_rows
0 2026-03-01 2026-03-31         9841378

--- Query 3: Availability (IS TRUE) ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   valid_pages
0       331045

--- Feature Engineering & The Leakage Trap ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Leaked Score (Precision): 1.00 <- Unrealistic! Model learned the math, not the pattern.
Honest Score (Precision): 0.91 <- Safe, no target leakage.


My 5 Features:

total_impressions: Knowable at the decision moment because past visibility is logged daily.

avg_position: Knowable at the decision moment from historical SERP ranking data.

total_clicks: Knowable at the decision moment from historical engagement logs.

has_gsc_access: Knowable at the decision moment via client setup state.

Trap Removed: true_ctr was removed as a feature because the target label was mathematically derived directly from it.

Limitation: Search metrics are highly volatile and context-dependent. A page dropping in CTR might not necessarily need a content refresh; it could be the result of a Google layout change (like adding AI Overviews to the top of the page) pushing traditional organic clicks down, which content changes cannot fix.

[x] 5 plain-words contract answers

[x] 3 verification queries (grain, row count/dates, availability with IS TRUE)

[x] 5 features max with "knowable because" lines

[x] The deliberate-leak experiment shown, explained, and removed

[x] 1 named limitation